In [ ]:
!python -m spacy download fr_core_news_md
!python -m spacy download en_core_web_md

In [ ]:

import pandas as pd
import os
import s3fs

import datetime
import json
import re
import tqdm

In [ ]:
import spacy

In [ ]:
import bertopic

In [ ]:
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/abstracts/2026-03-11_publis_with_abstract.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")
df0 

In [ ]:
df = df0.loc[(df0.abstract != "no abstract") & (df0.lang == 'en')]

In [ ]:


df["clean_text"] = df.apply(lambda row: re.sub(r"http\S+", "", str(row.abstract)).lower(), 1) # remove urls
df["clean_text"] = df.apply(lambda row: re.sub(r"_p_", "", str(row.clean_text)).lower(), 1)
df["clean_text"] = df.apply(lambda row: " ".join(filter(lambda x:x[0]!="@", row.clean_text.split())), 1) #remove mentioned names
df["clean_text"] = df.apply(lambda row: re.sub("’", "'", str(row.clean_text)).lower(), 1) #replace "’" by "'"
df["clean_text"] = df.apply(lambda row: re.sub("d\'|l\'|qu\'", "", str(row.clean_text)).lower(), 1) #replace "’" by "'"
df["clean_text"] = df.apply(lambda row: " ".join(re.sub("[^'a-zA-Zàâäéèêëïîôöùûüÿçß]+", " ", row.clean_text).split()), 1) #remove hashtag, arobase, HTML character
df["tokens"] = df.apply(lambda row: row.clean_text.split(),1) # 
df[["abstract", "clean_text", "tokens"]].head(3)



In [ ]:
df["len_txt"] = df.tokens.str.len()
df1 = df.loc[(df.len_txt >= 3)].reset_index().drop(columns=["index"])

print(f"Il y a {len(df1)} abstract. Le plus long fait {max(df1.len_txt)} caractères.")

In [ ]:
df1["text_id"] = "id_" + df1.index.astype(str) # on crée un id qu'on réutilisera plus bas



texts = df1.clean_text.to_list() #liste des textes nettoyés


# Embeddings

In [ ]:
!pip install sentence_transformers

In [ ]:


from sentence_transformers import SentenceTransformer
import pickle



In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Taille maximale de la séquence : ", model.get_max_seq_length())

In [ ]:


saved_embedding = True
if saved_embedding == True:
    with fs.open(f'{BUCKET_OUT}/Data_bso/outputs/embeddings/abstracts_english_embedding_raphrase-multilingual-MiniLM-L12-v2_2.pickle', 'rb') as pkl:
        embeddings = pickle.load(pkl)
else:
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
    with fs.open(f'{BUCKET_OUT}/Data_bso/outputs/embeddings/abstracts_english_embedding_raphrase-multilingual-MiniLM-L12-v2_2.pickle', 'wb') as pkl:
        pickle.dump(embeddings, pkl)
    



In [ ]:
import numpy as np
import umap

#librairies de visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:


fig, ax = plt.subplots(1, figsize=(14, 14))

fit = umap.UMAP(n_neighbors=15, min_dist=0.00, n_components=2, random_state=42, metric = 'cosine')
u = fit.fit_transform(embeddings)
sns.scatterplot(x=u[:,0], y=u[:,1])
#ax.set_title(f'n={n_neighbors}')




In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(14, 14))
nns = [2, 3, 4, 5, 10, 15, 20, 30, 100]
i, j = 0, 0
for n_neighbors in tqdm.tqdm(nns):
    fit = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.00, n_components=2, random_state=42, metric = 'cosine')
    u = fit.fit_transform(embeddings)
    sns.scatterplot(x=u[:,0], y=u[:,1],  ax=ax[j, i])
    ax[j, i].set_title(f'n={n_neighbors}')
    if i < 2: i += 1
    else: i = 0; j += 1

In [ ]:


umap_model = umap.UMAP(n_neighbors=4, 
                       n_components=15, 
                       min_dist=0.00,
                       metric='cosine',
                       random_state=42)



In [ ]:
import hdbscan

In [ ]:
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=50, 
                        min_samples= 5,
                        cluster_selection_method='eom', 
                        prediction_data=True)

In [ ]:


from bertopic.vectorizers import ClassTfidfTransformer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer



In [ ]:
#nltk_french_stopwords = stopwords.words('french')
#nlp = spacy.load('fr_core_news_md')
nlp = spacy.load("en_core_web_md")

In [ ]:
vectorizer_model = CountVectorizer(ngram_range=(1, 2),
                                   strip_accents='unicode',
                                    stop_words= [x for x in nlp.Defaults.stop_words])

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [ ]:


topic_model = bertopic.BERTopic(
    language="english",
    #embedding_model= model,    # Step 1 - Extract embeddings
    umap_model=umap_model,              # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,        # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,  # Step 4 - Tokenize topics
    ctfidf_model=ctfidf_model,          # Step 5 - Extract topic words
    calculate_probabilities=True,        
    verbose=True
)



In [ ]:


topics, probs = topic_model.fit_transform(texts, embeddings)



In [ ]:


print("Nombre de topics (moins les outliers) : ", len(set(topics))-1)



In [ ]:


#topic_labels = topic_model.generate_topic_labels(nr_words=10, topic_prefix=True, word_length=20, separator=", ")
#topic_model.set_topic_labels(topic_labels)
top_lab = topic_model.get_topic_info()
top_lab.head(15)



# Renommer les clusters

In [ ]:


topic_labels = topic_model.generate_topic_labels(nr_words=5, topic_prefix=True, word_length=20, separator="_")
topic_model.set_topic_labels(topic_labels)



In [ ]:


fig = topic_model.visualize_barchart(top_n_topics=46, n_words = 10)
fig

In [ ]:


topic_model.visualize_heatmap(custom_labels=True, n_clusters=14)



In [ ]:
# 1. Récupération des infos sur les documents avec .get_document_info(). On enlève les colonne superflues
dftexts = topic_model.get_document_info(texts).drop(columns=["CustomName", "Representative_document", "Representative_Docs", "Top_n_words"])
dftexts["text_id"] = "id_" + dftexts.index.astype(str)

In [ ]:
# 2. On fusionne dftexts avec df1 (dataframe de départ) pour récupérer le texte original et la date de publication
df2 = df1[["text_id", "abstract"]].merge(dftexts, on = ["text_id"], how = "left")
df2

In [ ]:


topic_no = 0 #numéro du topic à inspecter

# On filtre le dataframe pour ne garder que les textes appartenant au topic
dft = df2.loc[(df2["Topic"] == topic_no)]

dft

In [ ]:


print(dft.Representation.iloc[0]) ## J'imprime le label bertopic

for n, x in enumerate(dft.abstract.iloc[0:50]): #Et pour chaque text de dft 
    print("###########")
    print(x)

